> **📦 Optional dependencies**
>
> This article uses features from the pdstools `adm` extra. Install with your favorite package manager, e.g. `uv pip install "pdstools[adm]"`.


# Customer Profiles from ADM Naive Bayes Model Data

The predictor binning from ADM models can deliver great insights to marketing and business and provides transparency into the models. 

Both AGB and NB types of ADM models already provide overall predictor importance views like these:



In [1]:
# These lines are only for rendering in the docs, and are hidden through Jupyter tags
# Do not run if you're running the notebook seperately
# Hidden from doc by virtue of cell tags - in VSCode right-click on the bar to the left of this cell, edit cell tags, see metadata

import plotly.io as pio

pio.renderers.default = "notebook_connected"
import warnings

warnings.simplefilter("ignore", SyntaxWarning)

In [2]:
import os
from pdstools import ADMDatamart, demo_data
import polars as pl
pl.Config.set_tbl_rows(20)  

# Set ADM_DATA_PATH to a folder containing your model and predictor exports.
adm_data_path = os.environ.get("ADM_DATA_PATH")
dm = (
    ADMDatamart.from_ds_export(base_path=adm_data_path)
    if adm_data_path
    else demo_data(source="generated").datamart
)

fig = dm.plot.predictor_performance(top_n=10)
fig.update_layout(
    height=400, width=800, title="Feature Importance", xaxis_title="", yaxis_title=""
)

fig.show()

While helpful to understand how a model works, a global perspective on the model features is not of much use to understand who accepts what. For this type of insight, we need to dive deeper into how the model partitions the customers. The Pega ADM models can provide such information.

## Predictor Binning for Individual Models

The predictor binning for Baysian ADM models is available in Pega Prediction Studio and also in the downloadable, off-line model reports you can create with PDS tools. These reports are available for both the legacy R and the new Python versions of the toolkit.

You can create these reports directly from the PDS tools app, or work in an IDE, for more information see [the PDS Tools Documentation](https://pegasystems.github.io/pega-datascientist-tools/index.html).

You can easily recreate all or parts of this in code. For example, to recreate the binning reports use code like shown below. You'll need the model ID that you can pick up from Prediction Studio or from a quick analysis with PDS tools functions.

In [3]:
fig = dm.plot.predictor_binning(
    model_id="05085857-c52c-5daa-b51b-0af576a038af",
    predictor_name="IH.Web.Inbound.Clicked.pxLastOutcomeTime.DaysSince",
)
fig.update_layout(height=400, width=700, xaxis_title="")
fig.show()

You can also view these reports in an alternative way, focussing on how each of the bins pushes the propensity above or below the average. This perspective on the "lift" of each bin is similar to the yellow line in the above plot but emphasizes it more.

In [4]:
fig = dm.plot.binning_lift(
    model_id="05085857-c52c-5daa-b51b-0af576a038af",
    predictor_name="IH.Web.Inbound.Clicked.pxLastOutcomeTime.DaysSince",
)
fig.update_layout(height=400, width=700, xaxis_title="")
fig.show()

## Rolling up individual binning

While already giving more insight than global predictor importance, being specific to one individual ADM instance means you may have to browse through 100's of individual model reports to understand how a predictor relates to action response.

With the "BinAggregator" class in PDS Tools you can now "roll up" these binning views across actions and even across channels. For example, this could show that people who clicked recently are more likely to respond positively to Loan offers.

The rolling up is based on the predictor binning from ADM NB models. For a numeric predictor, we first create equi-distance or log-distance bins, for example, bins over days since the last click or a logarithmic scale for a heavily skewed predictor.

Then, the predictor binning of all models (in a certain group, issue perhaps) is mapped onto those target bins and the values are mapped proportional to the overlap between the model bin and the target bin. We map the *lift* values, not the *bin counts*. The bin counts are heavily dependent on channel both in absolute numbers as in the ratio between positives and negatives (success rate) so aggregating those does give meaningful results. The lift is an indication of how a certain predictor value range pushes the likelhood to accept up or down.




To illustrate this, see below. We have an equi-distance target binning for days since the last click, shown in red, from 0 to 90. From one of the models we have binning of the same predictor with a slightly different range (0–75) - shown in blue. The first bin of the target gets a weighted lift from source bins 1 and 2. The second target bin falls completely within the range of bin 2 of the source, so gets the exact same lift value. The target bins 3 and 4 are both sourced from bin 3. The last one is not fully covered, as you see reflected in the "BinCoverage" column in the table.

In [5]:
# For PDS tools example keep dm as above but the subset argument is important
myAggregator = dm.bin_aggregator

predictor_name = "IH.Web.Inbound.Clicked.pxLastOutcomeTime.DaysSince"
target = myAggregator.create_empty_numbinning(predictor_name, 4, minimum=0, maximum=90)
source = pl.DataFrame(
    {
        "ModelID": [1] * 3,
        "PredictorName": [predictor_name] * 3,
        "BinIndex": [1, 2, 3],
        "BinLowerBound": [0.0, 30.0, 60.0],
        "BinUpperBound": [30.0, 60.0, 75.0],
        # "BinSymbol" : ["20-25", "25-50", "50-75"],
        "Lift": [0.4, -0.1, 2.0],
        "BinResponses": [100, 1000, 400],
    }
)
myAggregator.plot_binning_attribution(source, target).update_layout(width=700)

The result of mapping the source binning onto this target is shown below. 
The resulting **Lift** is the average lift of the overlapping segments, weighted by overlap. It is not weighed by response count like we usually do for model performance etc, as this would skew the numbers heavily towards the positive bins (as generally the actions will be selected where the bins are scoring higher). The **BinResponses** is an indication of the number of responses (postive plus negative) for the bin (but is not used, only provided for additional insights). **BinCoverage** is the sum of the coverage by all the models for this new bin. It cannot be higher than the number of models (**Models**) - some models are empty and not taken into account at all, or they have a value range smaller than the combined binning.

In [6]:
myAggregator.combine_two_numbinnings(source, target)

PredictorName,BinIndex,BinLowerBound,BinUpperBound,BinSymbol,Lift,BinResponses,BinCoverage,Models
str,i64,f64,f64,str,f64,f64,f64,i32
"""IH.Web.Inbound.Clicked.pxLastO…",1,0.0,22.5,"""<22.5""",0.4,75.0,1.0,1
"""IH.Web.Inbound.Clicked.pxLastO…",2,22.5,45.0,"""<45.0""",0.066667,525.0,1.0,1
"""IH.Web.Inbound.Clicked.pxLastO…",3,45.0,67.5,"""<67.5""",0.6,700.0,1.0,1
"""IH.Web.Inbound.Clicked.pxLastO…",4,67.5,90.0,"""<90.0""",2.0,200.0,0.333333,1


So you can roll up days since the last click across all models in the default data set (or your own data when `ADM_DATA_PATH` is set) and visualize how recent interaction history affects propensity:

In [7]:
fig = myAggregator.roll_up(predictor_name, minimum=0, maximum=90)
fig.update_layout(height=300, width=600)
fig.show()

The above is a view across *all* models in *all* channels and may not be that meaningful. It shows how response propensity varies with the time since a customer's last click, but that in itself may not be that insightful.

It generally is much more useful to compare this distribution across different issues, groups or other dimensions of interest. For example, showing how time since the last click varies across action groups can be done with the following:

In [8]:
fig = myAggregator.roll_up(
    predictor_name, minimum=0, maximum=90, n=5, aggregation="Group"
)
fig.update_layout(height=500)
fig.show()

If you're interested in the underlying data rather than just the plot, use the 'return_df' argument - like in many of the PDS Tools plot functions.

In [9]:
myAggregator.roll_up(
    predictor_name, minimum=0, maximum=90, n=5, aggregation="Group", return_df=True
).head(20)

PredictorName,BinIndex,BinLowerBound,BinUpperBound,BinSymbol,Lift,BinResponses,BinCoverage,Models,Group
str,i64,f64,f64,str,f64,f64,f64,i32,str
"""IH.Web.Inbound.Clicked.pxLastO…",1,0.0,18.0,"""<18.0""",0.058389,7665.105882,3.0,3,""""""
"""IH.Web.Inbound.Clicked.pxLastO…",2,18.0,36.0,"""<36.0""",0.058389,7665.105882,3.0,3,""""""
"""IH.Web.Inbound.Clicked.pxLastO…",3,36.0,54.0,"""<54.0""",0.058389,7665.105882,3.0,3,""""""
"""IH.Web.Inbound.Clicked.pxLastO…",4,54.0,72.0,"""<72.0""",0.058389,7665.105882,3.0,3,""""""
"""IH.Web.Inbound.Clicked.pxLastO…",5,72.0,90.0,"""<90.0""",0.058389,7665.105882,3.0,3,""""""
"""IH.Web.Inbound.Clicked.pxLastO…",1,0.0,18.0,"""<18.0""",0.037912,1.1968e6,47.0,47,"""Collections"""
"""IH.Web.Inbound.Clicked.pxLastO…",2,18.0,36.0,"""<36.0""",0.022345,549788.718237,47.0,47,"""Collections"""
"""IH.Web.Inbound.Clicked.pxLastO…",3,36.0,54.0,"""<54.0""",0.013073,497924.788866,47.0,47,"""Collections"""
"""IH.Web.Inbound.Clicked.pxLastO…",4,54.0,72.0,"""<72.0""",0.000396,255819.352134,47.0,47,"""Collections"""


The boundaries of the bin intervals are by default created automatically, but can be given explicitly. For a numeric predictor you can choose a linear or logarithmic scale and define explicit boundaries. Below we split by Channel and define a few boundaries for days since the last click:

In [10]:
fig = myAggregator.roll_up(
    predictor_name,
    boundaries=[1, 7, 30, 90],
    n=8,
    distribution="log",
    aggregation="Channel",
)
fig.update_layout(height=300)
fig.show()

## Symbolic Predictors

For symbolic ('categorical') predictors the process is conceptually simpler. We first extract the symbols from the bin labels, then aggregate the lift for the symbols up. For both types, the aggregated lift is weighted proportional to the responses of the models.

While the way symbolic predictors are aggregated is very different, the process to roll them up is similar to that of numeric predictors. You can pass in some explicit symbols to consider, set the maximum and aggregate over some dimension like Issue or Group exactly as you can for numeric predictors.

In [11]:
fig = myAggregator.roll_up("Param.MessageLengthBand")
fig.update_layout(height=300, width=600)
fig.show()

## Multiple predictors at once

If you want to look at the aggregated lift of the top-5 predictors you can also do that. Instead of one predictor name, you can pass a list. You can even do this in combination with splitting on e.g. Group or Issue although this may be a little overwhelming.

Remember that you can always subset to a subset of the models when creating the **BinAggregator**.

In [12]:
top_predictors = sorted(list(set(dm.plot.predictor_performance(
    top_n=5,
    query=pl.col("PredictorCategory")== "Customer",  # skip the "IH" predictors, taking only the ones prefixed with "Customer."
    return_df=True,  
)['PredictorName'].to_list())))
 
fig = myAggregator.roll_up(top_predictors, n=6, aggregation="Group")
fig.update_layout(height=1000)
fig.for_each_annotation(
    lambda a: a.update(text=a.text.split(".")[-1])
)  # trick to show just the part of the predictor names after the dot

fig.show()